# 检查一份 fMRI 样例数据

本 Notebook 逐步读取一名 ADHD-200 受试者的预处理后静息态功能像。目标是学会从 NIfTI header 读取数据形状、空间分辨率、TR 和坐标变换，而不是从文件名猜参数。

首次运行需要网络下载数据；下载后 Nilearn 会缓存到本地。该影像已经过空间预处理，不要再次做运动校正或标准化。

## 1. 下载并载入单个受试者

`fetch_adhd(n_subjects=1)` 只请求一个受试者。`func_img` 是包含空间维度和时间维度的 NIfTI 图像对象；实际 voxel 数据可通过 `get_fdata()` 读取，但完整 4D 影像可能占用不少内存，所以先看元信息。

In [ ]:
from nilearn import datasets, image

dataset = datasets.fetch_adhd(n_subjects=1)
func_file = dataset.func[0]
func_img = image.load_img(func_file)
print(f"影像路径：{func_file}")
print(f"Python 图像对象：{type(func_img).__name__}")

## 2. 检查维度与采样信息

前三个维度表示空间网格；第四维是连续采集的 volume。`get_zooms()` 按维度给出空间体素尺寸和时间采样间隔。

In [ ]:
shape = func_img.shape
zooms = func_img.header.get_zooms()
spatial_unit, time_unit = func_img.header.get_xyzt_units()

print(f"维度 (x, y, z, time)：{shape}")
print(f"体素尺寸和 TR：{zooms}")
print(f"空间单位：{spatial_unit}")
print(f"时间单位：{time_unit}")
assert len(shape) == 4, f"预期 4D 功能像，实际形状为 {shape}"
print(f"时间点数：{shape[3]}")

**检查点：**当前图像有多少个 volume？TR 的单位是什么？若 TR 不合理或单位缺失，应回查采集协议和数据集文档，而不是自行猜测或默默替换数值。

## 3. 了解体素到世界坐标的变换

Affine 矩阵把图像体素坐标映射到物理空间坐标。方向和值取决于图像坐标系，不能仅靠矩阵外观推断左右方向；需要结合可视化和元数据核对。

In [ ]:
print(f"Affine to world coordinates:\n{func_img.affine}")
print("此样例已经过空间预处理；不要把它再次标准化。")

## 4. 显示一个 volume

显示中间时间点的三个正交切面。请观察视野中是否大致包含脑部，并记住：这只是初步目视，不等于完整的配准、头动或预处理质量控制。

In [ ]:
from nilearn import plotting

middle_volume = shape[3] // 2
plotting.plot_img(
    image.index_img(func_img, middle_volume),
    display_mode="ortho",
    title=f"ADHD-200 sample: volume {middle_volume}",
)

## 回顾

- 为什么功能像通常是四维的？
- 哪些信息来自 header，哪些问题还需要采集元数据或 QC 报告来回答？
- 本样例已经做过空间预处理；接下来要做的是时间序列清理和 ROI 分析，而不是再次标准化。